# Exploring `tarteel-ai/tlog` before running Muaalem inference

This notebook explores the [`tarteel-ai/tlog`](https://huggingface.co/datasets/tarteel-ai/tlog) dataset (Tarteel-app recitation logs, `clean` split, 411 parquet shards, gated — needs an HF token with access) **before** running the repo's `Muaalem` inference pipeline on it at scale.

It is split into two parts:

- **Part 1 — Exploration**: schema, filename structure, audio format, whether parquet shards are   shuffled or grouped by surah/ayah, the known per-ayah imbalance (from `total_counts.json`, produced   by `load_tarteel_log.ipynb` running a full pass over all 411 files), and a small stratified sample   of clips to listen to against their reference text.
- **Part 2 — Inference sketch**: wires a handful of the sampled clips into the existing `Muaalem` class   (`quran_muaalem.Muaalem`) end-to-end, as a design sketch for the eventual full-scale run — not a   production pipeline yet.

This notebook is written to run **either locally or in Colab**: it reads `HF_TOKEN` from the environment first, and falls back to `google.colab.userdata` only if running in Colab. It deliberately downloads only a handful of parquet files (not all 411) — the full-dataset aggregate stats already live in `total_counts.json`.

## Setup

In [ ]:
try:
    import quran_muaalem  # noqa: F401
except ImportError:
    %pip install -q quran-muaalem

try:
    import datasets, huggingface_hub, soundfile  # noqa: F401
except ImportError:
    %pip install -q "datasets[audio]>4.0.0" soundfile huggingface_hub matplotlib pandas

In [ ]:
import io
import json
import os
from pathlib import Path

from datasets import Audio, Dataset, concatenate_datasets
from huggingface_hub import HfApi, hf_hub_download
import soundfile as sf


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def get_hf_token() -> str:
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    if in_colab():
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
        if token:
            return token
    raise RuntimeError(
        "No HF_TOKEN found. Set it as an environment variable (`export HF_TOKEN=...`) "
        "or, in Colab, add it under Secrets and grant this notebook access."
    )


HF_TOKEN = get_hf_token()
DATASET_REPO = "tarteel-ai/tlog"
SPLIT = "clean"

# Where to cache downloaded parquet files + saved sample clips.
if in_colab():
    from google.colab import drive

    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/tlog_explore_cache")
else:
    CACHE_DIR = Path("./tlog_explore_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Point this at the total_counts.json produced by load_tarteel_log.ipynb
# (a full pass over all 411 files -> per-ayah clip count + total duration).
TOTAL_COUNTS_PATH = Path("total_counts.json")

print(f"Cache dir: {CACHE_DIR.resolve()}")
print(f"Total counts path: {TOTAL_COUNTS_PATH.resolve()} (exists: {TOTAL_COUNTS_PATH.exists()})")

## Part 1 — Exploration

### 1. Repo-level check: what splits/files actually exist

In [ ]:
api = HfApi(token=HF_TOKEN)

repo_files = api.list_repo_files(DATASET_REPO, repo_type="dataset")
data_files = sorted(f for f in repo_files if f.startswith("data/"))
non_data_files = sorted(f for f in repo_files if not f.startswith("data/"))

# `data/<split>-00000-of-00411.parquet` -> split name is the token before the first `-`
splits_found = sorted({Path(f).name.split("-")[0] for f in data_files})

print(f"Splits found under data/: {splits_found}")
print(f"Total parquet files under data/: {len(data_files)}")
print("First few:")
for f in data_files[:5]:
    print(" ", f)

print("\nNon-data repo files (dataset card, configs, etc.):")
for f in non_data_files:
    print(" ", f)

If `splits_found` contains more than just `clean` (e.g. a `noisy`/raw split), decide whether that matters for the eventual inference run before going further.

### 2. Schema + filename structure — download just a couple of files

In [ ]:
split_files = sorted(f for f in data_files if Path(f).name.startswith(f"{SPLIT}-"))
NUM_FILES = len(split_files)
print(f"{NUM_FILES} files in split '{SPLIT}'")

# Keep this small on purpose -- we only need enough files to check schema and
# whether shards are shuffled across the whole Quran (see step 4 below).
# Add more indices here (and re-run downstream cells) if the quality sample
# in step 6 can't find enough clips for its target ayat.
SAMPLE_FILE_INDICES = [0, 1]

local_paths = {}
for i in SAMPLE_FILE_INDICES:
    fname = f"{SPLIT}-{i:05d}-of-{NUM_FILES:05d}.parquet"
    local_paths[i] = hf_hub_download(
        repo_id=DATASET_REPO,
        filename=f"data/{fname}",
        repo_type="dataset",
        token=HF_TOKEN,
        cache_dir=str(CACHE_DIR / "hf_cache"),
    )
    print(f"Downloaded file {i}: {local_paths[i]}")

In [ ]:
sample_datasets = {
    i: Dataset.from_parquet(local_paths[i]).cast_column("audio", Audio(decode=False))
    for i in SAMPLE_FILE_INDICES
}
ds0 = sample_datasets[SAMPLE_FILE_INDICES[0]]

print("Features / schema:")
print(ds0.features)
print(f"\nRows in file {SAMPLE_FILE_INDICES[0]}: {len(ds0)}")

print("\nSample rows (audio bytes omitted):")
for row in ds0.select(range(5)):
    printable = {
        k: ({kk: vv for kk, vv in v.items() if kk != "bytes"} if k == "audio" else v)
        for k, v in row.items()
    }
    print(printable)

Look at the printed schema above: is there anything besides `audio` (e.g. a transcription, reciter/user id, riwayah/qiraat tag, timestamp)? That directly affects whether the Hafs-riwayah assumption used below (and by the rest of this repo) is safe for every clip.

### 3. Filename structure — what do the tokens after `surah_ayah` mean?

In [ ]:
def get_sura_aya_key(path):
    """Same parsing as load_tarteel_log.ipynb — first two `_`-separated tokens."""
    if not path:
        return None
    filename = os.path.basename(path)
    parts = filename.split("_")
    if len(parts) >= 3 and parts[0].isdigit() and parts[1].isdigit():
        return f"{parts[0]}_{parts[1]}"
    return None


print("Full filenames vs. parsed surah_ayah key (what do the remaining tokens encode?):")
for row in ds0.select(range(10)):
    path = row["audio"]["path"]
    print(f"  {path}  ->  {get_sura_aya_key(path)}")

### 4. Audio format — sample rate, channels, bit depth

In [ ]:
print("Decoded audio format for a few clips:")
for row in ds0.select(range(5)):
    audio = row["audio"]
    if not audio.get("bytes"):
        print(f"  {audio['path']}: <no bytes>")
        continue
    with io.BytesIO(audio["bytes"]) as f:
        info = sf.info(f)
    print(
        f"  {audio['path']}: sr={info.samplerate}, channels={info.channels}, "
        f"duration={info.duration:.2f}s, format={info.format}/{info.subtype}"
    )

`Muaalem` requires **16kHz mono** input. If the sample rate printed above isn't 16000, the inference sketch in Part 2 needs to resample.

### 5. Are the 411 files shuffled across the whole Quran, or grouped by surah/ayah?

In [ ]:
keys_per_file = {}
for i, ds in sample_datasets.items():
    keys = {get_sura_aya_key(r["path"]) for r in ds["audio"]}
    keys.discard(None)
    surahs = {int(k.split("_")[0]) for k in keys}
    keys_per_file[i] = keys
    print(
        f"File {i}: {len(ds)} rows, {len(keys)} unique ayat, "
        f"{len(surahs)} distinct surahs, range {min(surahs)}-{max(surahs)}"
    )

if len(SAMPLE_FILE_INDICES) >= 2:
    a, b = SAMPLE_FILE_INDICES[:2]
    overlap = keys_per_file[a] & keys_per_file[b]
    print(f"\nAyah-key overlap between file {a} and file {b}: {len(overlap)}")
    print(
        "-> Wide surah range + overlap in a couple of files suggests shards are shuffled "
        "across the whole Quran (a few files are a representative sample).\n"
        "-> Narrow/disjoint surah ranges suggest shards are grouped, and targeting specific "
        "ayat would require picking specific files, not just the first few."
    )

### 6. Known imbalance from `total_counts.json` (full 411-file pass, already computed)

In [ ]:
with open(TOTAL_COUNTS_PATH, "r", encoding="utf-8") as f:
    counts_data = json.load(f)

total_hours = sum(v["hours"] for v in counts_data.values())
total_clips = sum(v["count"] for v in counts_data.values())
per_ayah_counts = [v["count"] for v in counts_data.values()]

print(f"Ayat covered: {len(counts_data)} / 6236")
print(f"Total hours: {total_hours:.1f}")
print(f"Total clips: {total_clips}")
print(
    f"Per-ayah clip count: min={min(per_ayah_counts)}, "
    f"median={sorted(per_ayah_counts)[len(per_ayah_counts) // 2]}, "
    f"max={max(per_ayah_counts)}"
)

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Hours per surah (same chart as load_tarteel_log.ipynb, kept here for a self-contained view)
surah_hours = Counter()
for key, info in counts_data.items():
    surah_hours[int(key.split("_")[0])] += info.get("hours", 0)

surah_ids = list(range(1, 115))
hours_list = [surah_hours.get(i, 0) for i in surah_ids]

plt.figure(figsize=(20, 6))
plt.bar(surah_ids, hours_list, color="mediumpurple", edgecolor="indigo")
plt.title("Total Audio Duration per Surah (Hours)")
plt.xlabel("Surah Number")
plt.ylabel("Total Hours")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.xlim(0, 115)
plt.tight_layout()
plt.show()

In [ ]:
# Log-scale histogram of clip count per ayah -- makes the imbalance (7 to ~7800 clips
# per ayah) obvious in a way the per-surah aggregate above doesn't show.
plt.figure(figsize=(10, 5))
plt.hist(per_ayah_counts, bins=50)
plt.yscale("log")
plt.xscale("log")
plt.title("Distribution of clip count per ayah (log-log)")
plt.xlabel("Clips per ayah")
plt.ylabel("Number of ayat (log scale)")
plt.tight_layout()
plt.show()

**Takeaway**: any sampling, fine-tuning, or evaluation strategy over this dataset needs to account for this imbalance explicitly (e.g. cap clips per ayah, or weight by inverse frequency) — a uniform random sample will be dominated by Al-Fatiha and other very common ayat.

### 7. Stratified quality sample — listen to a few real clips against their reference text

In [ ]:
# Pick one low-count, one median-count, and the highest-count ayah as a small,
# representative sample to eyeball.
sorted_by_count = sorted(counts_data.items(), key=lambda kv: kv[1]["count"])
low_key = sorted_by_count[0][0]
median_key = sorted_by_count[len(sorted_by_count) // 2][0]
high_key = sorted_by_count[-1][0]
target_keys = {low_key, median_key, high_key}
print("Target ayah keys for quality sample:", target_keys)

In [ ]:
from quran_transcript import Aya

SAMPLE_OUT_DIR = CACHE_DIR / "quality_samples"
SAMPLE_OUT_DIR.mkdir(exist_ok=True)

combined = concatenate_datasets(list(sample_datasets.values()))

found = {k: [] for k in target_keys}
for row in combined:
    audio = row["audio"]
    key = get_sura_aya_key(audio["path"])
    if key in found and len(found[key]) < 3 and audio.get("bytes"):
        found[key].append(audio)

for key, clips in found.items():
    surah, ayah = (int(x) for x in key.split("_"))
    try:
        ref_text = Aya(surah, ayah).uthmani
    except Exception as e:
        ref_text = f"<could not load reference: {e}>"
    print(f"\n=== Ayah {key} ({len(clips)} clips found) — reference text ===\n{ref_text}")
    for idx, audio in enumerate(clips):
        out_path = SAMPLE_OUT_DIR / f"{key}_{idx}.wav"
        out_path.write_bytes(audio["bytes"])
        print(f"  saved {out_path}")

print(f"\nListen to the clips in {SAMPLE_OUT_DIR} and compare against the reference text above.")
if any(len(v) == 0 for v in found.values()):
    print(
        "\nNote: some target ayat had zero clips in the sampled files. If step 5 showed the "
        "shards are surah/ayah-grouped rather than shuffled, add more file indices to "
        "SAMPLE_FILE_INDICES above and re-run from there."
    )

When listening, check for: right ayah (path metadata is trustworthy), full recitation and not cut off mid-ayah, audible speech (not silence/noise-only), and a single speaker per clip.

### 8. Data quality summary (fill in after running the cells above)

- **Schema**: columns found besides `audio` — ...
- **Riwayah/qiraat**: is it tagged in the data, or is Hafs a safe assumption for the whole dataset? — ...
- **Sample rate**: native sample rate observed — ... (resample to 16kHz needed? yes/no)
- **Shard layout**: shuffled across the Quran vs. grouped — ...
- **Filename tokens**: what `parts[2:]` in the filename encode (reciter/user id, recording id, etc.) — ...
- **Quality sample notes**: anything off in the clips listened to (truncated, silent, wrong ayah, multiple speakers) — ...
- **Filters to apply before inference**: e.g. min/max duration, drop failed decodes — ...


## Part 2 — Inference sketch

This section is a **design sketch**, not a production run: it wires the small quality sample saved above into the repo's existing `Muaalem` inference class, following the same pattern as `tests/test_muaalem_infrence.py`. It assumes Hafs riwayah — revisit that assumption based on what the schema check in Part 1 found.

The full-scale run (all 411 files, checkpointed per-shard like `load_tarteel_log.ipynb` does) is intentionally **not** implemented here — it should be its own focused pass once the findings above are filled in (confirmed schema, riwayah, sample rate, and filtering rules).

In [ ]:
from quran_transcript import MoshafAttributes, quran_phonetizer

# NOTE: assumes Hafs riwayah for every clip -- confirm this against the schema findings
# above before relying on it for anything beyond this small sketch.
DEFAULT_MOSHAF = MoshafAttributes(
    rewaya="hafs",
    madd_monfasel_len=2,
    madd_mottasel_len=4,
    madd_mottasel_waqf=4,
    madd_aared_len=2,
)


def build_reference(surah: int, ayah: int, moshaf: MoshafAttributes = DEFAULT_MOSHAF):
    uthmani = Aya(surah, ayah).uthmani
    return quran_phonetizer(uthmani, moshaf, remove_spaces=True)

In [ ]:
TARGET_SR = 16000


def load_wave_16k(wav_path: Path):
    wave, sr = sf.read(str(wav_path), dtype="float32", always_2d=False)
    if wave.ndim > 1:
        wave = wave.mean(axis=1)  # downmix to mono
    if sr != TARGET_SR:
        import librosa

        wave = librosa.resample(wave, orig_sr=sr, target_sr=TARGET_SR)
    return wave

In [ ]:
import torch

from quran_muaalem import Muaalem

muaalem = Muaalem(device="cuda" if torch.cuda.is_available() else "cpu")

waves, refs, keys = [], [], []
for wav_path in sorted(SAMPLE_OUT_DIR.glob("*.wav")):
    key = "_".join(wav_path.stem.split("_")[:2])
    surah, ayah = (int(x) for x in key.split("_"))
    waves.append(load_wave_16k(wav_path))
    refs.append(build_reference(surah, ayah))
    keys.append(key)

outs = muaalem(waves, refs, sampling_rate=TARGET_SR)
for key, wav_path, out in zip(keys, sorted(SAMPLE_OUT_DIR.glob("*.wav")), outs):
    print(f"{wav_path.name} (ayah {key}): predicted phonemes = {out.phonemes.text}")

### Next steps for the full-scale run (not implemented here)

1. Apply the filters decided in the Part 1 summary (min/max duration, drop failed audio decodes — `load_tarteel_log.ipynb`'s `process_parquet_file` already try/excepts `sf.info` failures).
2. Stream all 411 files one at a time (as `load_tarteel_log.ipynb` does) rather than downloading them all up front.
3. Batch clips into `Muaalem(waves, refs, sampling_rate=16000)` calls — the model needs ~1.5GB of GPU memory (per the repo README), so batch size can likely be generous on any modern GPU.
4. Persist `MuaalemOutput`s incrementally per parquet-file-shard, reusing `load_tarteel_log.ipynb`'s `processed_{file_index}.json`-style checkpoint pattern so a long run survives interruptions and can resume.
5. Decide what to do with the results — aggregate error rates per ayah/surah, per apparent reciter/user, export mismatches for manual review, etc.